# Flood LSTM — Training & Evaluation

Trains a 2-layer LSTM for 24/48/72-hour flood discharge forecasting.

**Output:** `models/flood_lstm.pt` — loaded by `backend/app/ml/flood_lstm.py`

## Data
Provide a CSV with columns: `datetime, rainfall_mm, river_level_m, gauge_id, basin`  
Sources: CWC India daily reports, WRIS, or synthetic data (use `--synthetic` flag in training script).

## Quick start (synthetic)
```bash
python ml/scripts/train_flood_lstm.py --synthetic --output models/flood_lstm.pt
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from app.ml.flood_lstm import FloodLSTM, FloodForecast

OUTPUT_PATH = Path('../models/flood_lstm.pt')
WINDOW = 7   # 7 time steps of history
FEATURES = 2  # [rainfall_mm, river_level_m]

In [ ]:
# ── Generate synthetic training data ──────────────────────────
np.random.seed(42)
N = 3000
t = np.linspace(0, 8*np.pi, N)
rainfall = np.clip(2*np.sin(t) + np.random.randn(N)*0.5 + 1, 0, None).astype(np.float32)
level    = np.clip(0.5*np.cumsum(rainfall*0.01) % 5 + np.random.randn(N)*0.1, 0, 10).astype(np.float32)

plt.figure(figsize=(14,3))
plt.subplot(1,2,1); plt.plot(rainfall[:200]); plt.title('Rainfall (mm)')
plt.subplot(1,2,2); plt.plot(level[:200], color='blue'); plt.title('River level (m)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Build sliding windows ─────────────────────────────────────
X, y = [], []
for i in range(len(level) - WINDOW - 3):
    X.append(np.stack([rainfall[i:i+WINDOW], level[i:i+WINDOW]], axis=1))
    y.append(level[i+WINDOW : i+WINDOW+3])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
print(f'X: {X.shape}  y: {y.shape}')

split = int(len(X)*0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

In [ ]:
# ── Train ─────────────────────────────────────────────────────
model = FloodLSTM(input_size=FEATURES, hidden_size=48)
metrics = model.train_model(X_train, y_train, X_val, y_val, epochs=25)
print(f"Val RMSE: {metrics['rmse']:.4f}   MAE: {metrics['mae']:.4f}")

In [ ]:
# ── Visual check: predicted vs actual on validation set ───────
model_torch = model.model
model_torch.eval()
with torch.no_grad():
    val_pred = model_torch(torch.tensor(X_val)).numpy()

plt.figure(figsize=(14,4))
plt.plot(y_val[:100, 0], label='Actual 24h level', alpha=0.8)
plt.plot(val_pred[:100, 0], label='Predicted 24h level', alpha=0.8)
plt.legend(); plt.title('FloodLSTM: 24h forecast vs actual'); plt.tight_layout(); plt.show()

In [ ]:
# ── Save ──────────────────────────────────────────────────────
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(OUTPUT_PATH)
print(f'Saved → {OUTPUT_PATH}')
print(f'Set env: FLOOD_LSTM_CHECKPOINT={OUTPUT_PATH.resolve()}')

In [ ]:
# ── Smoke-test inference ──────────────────────────────────────
loaded = FloodLSTM(checkpoint=OUTPUT_PATH)
sample = np.array(X_val[0], dtype=np.float32)  # shape (7, 2)
result: FloodForecast = loaded.predict(sample)
print(f'24h: {result.forecast_24h:.3f}  48h: {result.forecast_48h:.3f}  72h: {result.forecast_72h:.3f}')
print(f'Risk tier: {result.risk_tier}  Latency: {result.latency_ms:.1f}ms')